# 사전학습과 전이학습
모델을 매번 처음부터 학습하는 것은 막대한 연산비용이 따르나. 따라서, **사전학습**된 모델을 가져와 필요에 맞게 **전이학습**하여 사용한다.
- 사전학습 : 초중고 12년 + 대학 교육
- 전이학습 : 석박사 + 직무연수
- 파인튜닝 : 실제업무 경험

### 1) 모델을 필요에 맞게 사용하는 방법
|도전순서|방법|모델변형|사용예시|
|---|---|---|---|
|1|프롬프트|x|기본API+지시|
|2|임베딩 활용|x|검색/RAG|
|3|파인튜닝|o|분류기 등|

### 2) 임베딩 추출 vs. 파인튜닝
입문 단계나 데이터 자원이 적을 때는 **임베딩 추출**부터 시작하는 것이 합리적임.
|구분|임베딩 추출|파인튜닝|
|---|---|---|
|모델변형|사전학습 그대로|조금씩 조정|
|학습영역|위에 얹은 작은 분류기 정도만|모델 가중치(일부/전체)
|비용·속도|매우 가볍고 빠름|무겁고 느림|
|성능|충분|높음|
|비유|활용문서를 보고 답변|문서자체를 암기해서 답변|

### 3) 임베딩 활용
- **분류** : 임베딩+분류기(로지스틱 회귀 등) - 숫자 벡터를 기반으로 "어떤 유형이 확률이 높은지" 점수를 매김
- **유사 검색** : 질문 임베딩과 가까운 문서 임베딩 출력

# 실습: 임베딩 + 작은 분류기
거대 사전학습 임베딩은 그대로 쓰고, 위에 작은 분류기만 학습하는 전이학습

In [3]:
# sentence-transformers: 문장을 '의미 벡터(임베딩)'로 바꿔 주는 라이브러리. (코랩 기본 미설치)
# scikit-learn은 코랩 기본 설치.
# !pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer  # 인코더(문장 → 임베딩)
from sklearn.linear_model import LogisticRegression    # 위에 얹을 가벼운 분류기

# 의미 벡터를 뽑는 임베딩
# jhgan/ko-sroberta-multitask: 한국어 문장 임베딩 대표 사전학습 모델. HF 확인 후 변경 가능
emb = SentenceTransformer('jhgan/ko-sroberta-multitask')

# 문의 예시 5개와 정답 유형.
texts  = ['배송 언제 와요', '환불 해주세요', '품절 재입고 언제', '환불 가능한가요', '배송 며칠']
labels = ['배송문의', '환불', '재입고', '환불', '배송문의']

# 문장 인코딩
X = emb.encode(texts) # encode: 각 문장을 고정 길이 의미 벡터로 변환
# 분류기 학습
clf = LogisticRegression().fit(X, labels) # fit: 임베딩 X와 정답 labels의 관계 학습
# 예측
print(clf.predict(emb.encode(['언제 도착하나요'])))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['배송문의']


-> 학습에 없던 새 문장 `언제 도착하나요`도 의미가 가까운 `배송문의`로 분류됨

In [ ]:
# 임베딩 벡터가 어떻게 생겼는지 확인해보기
X

array([[-0.13282767,  0.03683304, -0.04156096, ...,  0.3769517 ,
         0.37753692, -0.48844957],
       [-0.33453226, -0.07264644,  0.561424  , ...,  0.41928765,
        -0.08357862, -0.41719976],
       [-0.5536006 ,  0.06714918, -0.00431857, ..., -0.12313648,
         0.27998587, -0.9310398 ],
       [-1.2020619 , -0.07914428,  0.5527945 , ...,  0.7141979 ,
         0.0673717 , -0.42990744],
       [-0.00325765,  0.0092417 ,  0.18756816, ...,  0.30449238,
         0.7980114 , -0.10144342]], dtype=float32)

In [4]:
print(clf.predict(emb.encode(['환불 해달라구요'])))

['환불']


# 실습: 사전학습 임베딩으로 전이학습 실전
임베딩모델(거대 사전학습)은 그대로 두고, 위에 가벼운 분류기만 얹어 쇼핑몰 고객 문의를 분류해 봅니다.  

### 0. 준비

In [5]:
# !pip install -q sentence-transformers scikit-learn pandas
# sentence-transformers: 문장을 '의미 벡터(임베딩)'로 바꿔주는 라이브러리
# scikit-learn: 전통 머신러닝(여기선 로지스틱 회귀 + 평가) 라이브러리

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

from sentence_transformers import SentenceTransformer  # 임베딩 모델 클래스
from sklearn.linear_model import LogisticRegression    # 가벼운 분류기
from sklearn.model_selection import train_test_split   # 데이터를 학습/평가로 분리
from sklearn.metrics import accuracy_score, classification_report  # 정확도 계산

ROOT = Path('/content/drive/MyDrive/kt cloud tech up/gen-ai'); DATA = ROOT / 'data'

Mounted at /content/drive


In [7]:
df = pd.read_csv(DATA / 'product_inquiries.csv') # 제품에 대한 고객문의 데이터
df.head(3)

,inquiry_id,product_id,product_name,inquiry_topic,inquiry_text,created_at
0,Q00001,P00071,속건 마이크로화이버 타월 3매,규격,속건 마이크로화이버 타월 3매 실제 사이즈(가로세로높이)가 어떻게 되나요? 설치 공...,2026-04-30 03:44
1,Q00002,P00061,스탠다드 후드 집업,소재,스탠다드 후드 집업 소재가 비치는 편인가요? 안에 이너 받쳐 입어야 할까요?,2026-03-29 17:53
2,Q00003,P00075,조립식 3단 수납 선반,내구성,오래 써도 녹슬거나 휘지 않나요? 내구성이 어떤지 궁금해요.,2026-03-14 01:27


In [12]:
print(set(df['inquiry_topic']))

{'색상', '세탁관리', '소음', '관리', '구성', '재입고', '착화감', '전력', '용량', '사용법', '사이즈', '규격', '호환', '내구성', '설치', '배송', '소재', '선물'}


In [13]:
df['inquiry_topic'].value_counts()

,count
inquiry_topic,
재입고,35
소재,34
사이즈,29
배송,28
색상,26
세탁관리,13
구성,12
용량,10
사용법,8


In [15]:
# '소음' 카테고리는 학습/테스트 분리가 불가능하여 제거
df_removed = df[df['inquiry_topic'] != '소음']
# 정답 라벨(문의 유형) 분리
y = df_removed['inquiry_topic']

### 1. 사전학습 임베딩 모델로 고객문의 벡터화

In [16]:
# ko-sroberta-multitask: 한국어 문장 임베딩 모델(문장 → 고정 길이 벡터)
embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
# encode: 각 문장을 의미 벡터로 변환. show_progress_bar=True로 진행률 표시
X = embedder.encode(df_removed['inquiry_text'].tolist(), show_progress_bar=True)
print('임베딩 shape:', X.shape)   # (문장 수, 임베딩 차원) — 문장마다 같은 길이의 벡터

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

임베딩 shape: (219, 768)


In [17]:
# 임베딩 벡터 눈으로 보기
X[0,:]

array([ 2.91730333e-02, -3.93807828e-01,  3.79233778e-01, -5.27893364e-01,
       -2.39200592e-01,  4.60373789e-01,  3.52300286e-01,  1.76110163e-01,
       -1.62921667e-01,  5.68526573e-02,  2.22033173e-01, -3.30315053e-01,
        1.59334198e-01,  4.83024746e-01,  1.79261088e-01, -2.50037044e-01,
        3.04582238e-01,  4.87586290e-01,  5.39262116e-01, -3.24356779e-02,
        2.03024652e-02, -2.91040719e-01,  5.61755598e-01,  2.59901077e-01,
       -3.03350657e-01,  3.28539938e-01, -2.78956532e-01, -1.48321196e-01,
        2.30091795e-01, -2.08543718e-01, -4.02522296e-01,  2.31218338e-01,
       -2.03904286e-01,  3.15001100e-01,  7.91706085e-01,  9.72102359e-02,
       -7.83789679e-02, -5.22331148e-03, -1.95399389e-01,  2.00488836e-01,
        1.87139660e-01,  5.09224713e-01, -1.20343745e-01,  5.34986854e-02,
        4.44655687e-01,  1.24120653e-01,  9.54555869e-01, -1.85351476e-01,
       -2.89741397e-01, -3.47897977e-01,  2.99083710e-01, -1.94825575e-01,
       -3.24681662e-02,  

### 2. 임베딩 위에 가벼운 분류기
* 데이터를 학습 70% / 평가 30%로 분리. random_state=42=매번 같은 분할(재현성)
* stratify=y=라벨 비율을 학습/평가에 동일하게 유지(한쪽에 특정 유형이 몰리지 않게)

In [19]:
# 학습 데이터 분리
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# 모델 학습
clf = LogisticRegression(max_iter=1000)   # max_iter=1000: 학습이 수렴할 때까지 충분히 반복
clf.fit(Xtr, ytr)
# 평가셋을 예측해 정답과 비교한 정확도(0~1). round(...,3)=소수 셋째 자리로 반올림
print('정확도:', round(accuracy_score(yte, clf.predict(Xte)), 3))

정확도: 0.985


In [23]:
# 어떤 유형이 약한지(경계 사례)
pred = clf.predict(Xte)
print(classification_report(yte, pred, zero_division=0))

              precision    recall  f1-score   support

          관리       1.00      1.00      1.00         1
          구성       1.00      1.00      1.00         4
          규격       0.50      1.00      0.67         1
         내구성       1.00      1.00      1.00         1
          배송       1.00      1.00      1.00         8
         사용법       1.00      0.50      0.67         2
         사이즈       1.00      1.00      1.00         9
          색상       1.00      1.00      1.00         8
          선물       1.00      1.00      1.00         1
          설치       1.00      1.00      1.00         1
        세탁관리       1.00      1.00      1.00         4
          소재       1.00      1.00      1.00        10
          용량       1.00      1.00      1.00         3
         재입고       1.00      1.00      1.00        10
          전력       1.00      1.00      1.00         1
         착화감       1.00      1.00      1.00         1
          호환       1.00      1.00      1.00         1

    accuracy              

-> 전부 1.0인데 `규격`은 0.5로 낮음. 어떤 유형이 약점인지 확인

# 문의 로그 분류 - 캐싱·신뢰도 라우팅·모델 비교
① 분류 → ② 캐싱 → ③ 신뢰도 라우팅 → ④ 모델 비교

### 1. 문장 임베딩 + 분류
* 임베딩 모델 그대로 사용 embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')

In [24]:
df = pd.read_csv(DATA / 'cs_inquiries.csv')
df.head(3)

,inquiry_id,received_at,channel,customer_id,product_id,inquiry_type,inquiry_text,status
0,C00001,2026-04-06 21:52,앱,U12225,P00031,상품정보,머그컵 전자레인지 사용 가능한지 궁금합니다.,접수
1,C00002,2026-05-11 11:00,카카오톡상담,U63093,P00028,반품,"선물하려고 산 디퓨저인데, 받는 분이 이미 가지고 있다고 해서요. 다시 보내드리려면...",처리중
2,C00003,2026-04-07 16:37,웹,U49673,P00001,상품정보,"퀸 사이즈 침대 커버, 상세 페이지에 보이는 패턴 외에 다른 패턴은 없나요?",처리중


In [28]:
from collections import Counter
print(Counter(df['inquiry_type']))

Counter({'반품': 36, '교환': 35, '주문결제': 34, '회원멤버십': 30, '취소': 28, '기타': 27, '상품정보': 25, '재입고': 25, '배송문의': 20, '환불': 20})


In [29]:
# encode: 각 문장을 의미 벡터로 변환. show_progress_bar=True로 진행률 표시
X = embedder.encode(df['inquiry_type'].tolist(), show_progress_bar=True)
y = df['inquiry_type']            # 정답 라벨(문의 유형)
print('임베딩 shape:', X.shape)   # (문장 수, 임베딩 차원) — 문장마다 같은 길이의 벡터

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

임베딩 shape: (280, 768)


In [30]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)         # 임베딩 위 '가벼운' 분류기만 학습
pred = clf.predict(Xte)
print('전이학습 베이스라인 정확도:', round(accuracy_score(yte, pred), 3))
print(classification_report(yte, pred, zero_division=0))      # 어떤 유형이 약한지(경계 사례)

전이학습 베이스라인 정확도: 1.0
              precision    recall  f1-score   support

          교환       1.00      1.00      1.00        10
          기타       1.00      1.00      1.00         8
          반품       1.00      1.00      1.00        11
        배송문의       1.00      1.00      1.00         6
        상품정보       1.00      1.00      1.00         8
         재입고       1.00      1.00      1.00         8
        주문결제       1.00      1.00      1.00        10
          취소       1.00      1.00      1.00         8
          환불       1.00      1.00      1.00         6
       회원멤버십       1.00      1.00      1.00         9

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84



### 2. 캐싱
한 번 만든 벡터를 `.npy`로 저장하여 두 번 인코딩하지 않고 즉시 로드함.

In [31]:
import time, numpy as np
cache = ROOT / 'cache' / '010_cs_emb_ko-sroberta.npy'
cache.parent.mkdir(parents=True, exist_ok=True)

if cache.exists():
    t = time.time(); X = np.load(cache)
    print(f'캐시 로드: {time.time()-t:.3f}s (재계산 0)')      # 두 번째 실행부터: 0.000s
else:
    emb = SentenceTransformer('jhgan/ko-sroberta-multitask')
    X = emb.encode(df['inquiry_text'].tolist()); np.save(cache, X)
    print('인코딩 후 저장 — 다음 실행부터는 캐시 사용')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

인코딩 후 저장 — 다음 실행부터는 캐시 사용


### 3. 신뢰도 임계값 라우팅
모델 신뢰도 점수가 낮으면 상담사에게 전달. (오답 자동 처리를 막는 운영 패턴임.)

In [32]:
proba = clf.predict_proba(Xte)              # 각 유형일 확률
pred  = clf.classes_[proba.argmax(1)]
conf  = proba.max(1)                         # 가장 높은 확률 = 신뢰도
TH = 0.6                                      # 임계값
auto = conf >= TH
print(f'자동 분류 {auto.sum()}건 — 정확도 {accuracy_score(np.array(yte)[auto], pred[auto]):.3f}')
print(f'상담사 에스컬레이션(신뢰도<{TH}) {(~auto).sum()}건')

자동 분류 84건 — 정확도 1.000
상담사 에스컬레이션(신뢰도<0.6) 0건


In [47]:
text = '입어보기만 했는데 사이즈가 안 맞아서 바꾸고 싶은데 가능한가요?'
ans = clf.predict_proba(emb.encode([text]))
print(clf.classes_[ans.argmax(1)])
print(ans.max(1))

['재입고']
[0.45289606]


-> 0.6 이하의 신뢰도이므로 상담사 연결이 필요함

### 4. 임베딩 모델 비교

In [49]:
for name in ['jhgan/ko-sroberta-multitask', 'jhgan/ko-sbert-nli']:
    emb = SentenceTransformer(name)
    X = emb.encode(df['inquiry_text'].tolist())
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    acc = accuracy_score(yte, LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict(Xte))
    print(f'{name}: 정확도 {acc:.3f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

jhgan/ko-sroberta-multitask: 정확도 0.786


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

jhgan/ko-sbert-nli: 정확도 0.679


-> `ko-sroberta-multitask`(0.786)가 `ko-sbert-nli`(0.679)보다 높다.   
임베딩 품질이 곧 분류 성능이므로, 모델 선택이 중요(좋은 모델이 보통 더 무거움—정확도·속도 트레이드오프).